# ภาพที่ 4.6 — ตัวอย่างผลระดับช่องกริด ด้วยโมเดลรอบที่ 12 (v7) + SAHI 1024
**ก่อนรัน:** Add Input → Notebook output ของ `train_v7_finetune` (ที่มี `v7_best_sofar.pt`) · Secrets → `ROBOFLOW_API_KEY` · Accelerator: GPU T4 (ถ้าไม่เปิดก็รันได้แต่ช้ากว่า ~10 นาที)
กด **Run All** แล้วโหลด `fig46.zip` จากแท็บ Output

In [ ]:
# 1) ติดตั้ง + หาโมเดล + โหลดชุด test จาก Roboflow (version 2)
!pip -q install ultralytics sahi roboflow 2>&1 | tail -1
import os, glob, shutil, numpy as np, pandas as pd, torch
from PIL import Image, ImageOps, ImageDraw
K = '/kaggle/input'
def find(name, maxdepth=6):
    for d in range(1, maxdepth + 1):
        hit = sorted(glob.glob(f'{K}/' + '*/' * (d - 1) + name))
        if hit: return hit[0]
V7_PT = find('v7_best_sofar.pt')
assert V7_PT, 'ไม่เจอ v7_best_sofar.pt — Add Input ผลของโน้ตบุ๊ก train_v7_finetune ก่อน'
DEV = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('โมเดล:', V7_PT, '| device:', DEV)

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY').strip().strip('"')
RF_DIR = '/tmp/rf_v2'
if not os.path.exists(RF_DIR):
    Roboflow(api_key=KEY).workspace('attawit-madsait').project('weed-detect-l2bhf').version(2).download('yolov8', location=RF_DIR)
IMGS = sorted(glob.glob(f'{RF_DIR}/test/images/*'))
print('ภาพ test:', len(IMGS), '(ควรเป็น 140)')

In [ ]:
# 2) ทำนายด้วย SAHI 1024 (ค่าเดียวกับในเล่ม) แล้วคิดผลระดับช่องกริด 12 คอลัมน์ · conf ≥ 0.25 ต่อภาพ
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
det = None
for mt in ('ultralytics', 'yolov8'):
    try:
        det = AutoDetectionModel.from_pretrained(model_type=mt, model_path=V7_PT, confidence_threshold=0.01, device=DEV, image_size=1024); break
    except Exception as e: err = e
if det is None: raise err

def gt_boxes(p, W_, H_):
    lb = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'; out = []
    for l in (open(lb).read().splitlines() if os.path.exists(lb) else []):
        v = l.split()
        if len(v) == 5:
            _, cx, cy, w, h = map(float, v); out.append([(cx-w/2)*W_, (cy-h/2)*H_, (cx+w/2)*W_, (cy+h/2)*H_, 1.0])
        elif len(v) >= 7 and len(v) % 2 == 1:                 # เผื่อเป็น polygon
            xs, ys = list(map(float, v[1::2])), list(map(float, v[2::2]))
            out.append([min(xs)*W_, min(ys)*H_, max(xs)*W_, max(ys)*H_, 1.0])
    return np.array(out, np.float32).reshape(-1, 5)

def grid_mask(boxes, W_, H_, cols=12, conf_th=0.0):          # นิยามเดียวกับโน้ตบุ๊ก v7
    cell = W_ / cols; nr = max(1, round(H_ / cell)); ch = H_ / nr
    m = np.zeros((nr, cols), bool)
    for x1, y1, x2, y2, c in np.asarray(boxes).reshape(-1, 5):
        if c < conf_th: continue
        m[int(max(y1, 0)//ch):int(min(y2, H_-1)//ch)+1, int(max(x1, 0)//cell):int(min(x2, W_-1)//cell)+1] = True
    return m

rows, CACHE = [], {}
for i, p in enumerate(IMGS):
    im = np.asarray(ImageOps.exif_transpose(Image.open(p)).convert('RGB')); H_, W_ = im.shape[:2]
    r = get_sliced_prediction(im, det, slice_height=1024, slice_width=1024, overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0)
    d = np.array([[q.bbox.minx, q.bbox.miny, q.bbox.maxx, q.bbox.maxy, q.score.value] for q in r.object_prediction_list], np.float32).reshape(-1, 5)
    g = grid_mask(gt_boxes(p, W_, H_), W_, H_); dm = grid_mask(d, W_, H_, 12, 0.25)
    CACHE[p] = (g, dm)
    TP, FP, FN = int((g & dm).sum()), int((~g & dm).sum()), int((g & ~dm).sum())
    P = TP/max(TP+FP, 1); R = TP/max(TP+FN, 1)
    rows.append(dict(img=p, ภาพ=os.path.basename(p)[:40], วัชพืชจริง=round(g.mean()*100, 1), TP=TP, FP=FP, FN=FN,
                     P=round(P*100, 1), R=round(R*100, 1), F1=round(2*P*R/max(P+R, 1e-9)*100, 1)))
    if i % 20 == 0: print(f'{i+1}/{len(IMGS)}')
DF = pd.DataFrame(rows)
tp, fp, fn = DF.TP.sum(), DF.FP.sum(), DF.FN.sum(); P = tp/(tp+fp); R = tp/(tp+fn)
print(f'ทั้งชุด: P {P*100:.1f} R {R*100:.1f} F1 {2*P*R/(P+R)*100:.1f}  (ในเล่ม: 85.7 / 93.2 / 89.3)')

In [ ]:
# 3) เลือกภาพตัวแทน: วัชพืชปานกลาง (20–50% ของช่อง) · มีครบทั้ง 3 สี · F1 ใกล้ค่าเฉลี่ยทั้งชุด → ทำภาพให้เลือก 4 ภาพ
F1_ALL = 2*P*R/(P+R)*100
C = DF[(DF['วัชพืชจริง'].between(20, 50)) & (DF.TP > 0) & (DF.FP > 0) & (DF.FN > 0)].copy()
if len(C) < 4: C = DF[(DF.TP > 0)].copy()
C['ห่างจากค่าเฉลี่ย'] = (C.F1 - F1_ALL).abs()
C = C.sort_values('ห่างจากค่าเฉลี่ย').head(4)
display(C.drop(columns=['img']))

COL = {'TP': (27, 175, 122, 120), 'FP': (235, 104, 52, 140), 'FN': (42, 120, 214, 150)}
OUTD = '/kaggle/working/fig46'; shutil.rmtree(OUTD, ignore_errors=True); os.makedirs(OUTD)
for n, (_, row) in enumerate(C.iterrows(), 1):
    g, dm = CACHE[row.img]
    base = ImageOps.exif_transpose(Image.open(row.img)).convert('RGBA'); W_, H_ = base.size
    cell = W_ / 12; ch = H_ / g.shape[0]
    over = Image.new('RGBA', base.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(over)
    lw = max(2, W_ // 800)
    for r_ in range(g.shape[0]):
        for c_ in range(12):
            box = [c_*cell, r_*ch, (c_+1)*cell, (r_+1)*ch]
            kind = 'TP' if g[r_, c_] and dm[r_, c_] else 'FP' if dm[r_, c_] else 'FN' if g[r_, c_] else None
            if kind: dr.rectangle(box, fill=COL[kind])
            dr.rectangle(box, outline=(255, 255, 255, 110), width=lw)
    out = Image.alpha_composite(base, over).convert('RGB'); out.thumbnail((2000, 2000))
    out.save(f'{OUTD}/grid_eval_example_{n}.jpg', quality=92)
    print(f'grid_eval_example_{n}.jpg  ← {row["ภาพ"]} | วัชพืชจริง {row["วัชพืชจริง"]}% · P {row.P} R {row.R} F1 {row.F1}')
    display(out.resize((out.width // 2, out.height // 2)))
DF.drop(columns=['img']).to_csv(f'{OUTD}/grid_per_image_v7.csv', index=False)
shutil.make_archive('/kaggle/working/fig46', 'zip', OUTD)
print('เสร็จ — โหลด fig46.zip จากแท็บ Output แล้วเลือกภาพที่ชอบ 1 ภาพ')